## Output Parsing

Language models output text. But there are times where you want to get more structured information than just text back

Output parsers are classes that help structure language model responses. There are two main methods an output parser must implement:

- **Get format instructions**: A method which returns a string containing instructions for how the output of a language model should be formatted.
- **Parse**: A method which takes in a string (assumed to be the response from a language model) and parses it into some structure.

- Output Parsing
    - StrOutputParser
    - JsonOutputParser
    - CSV Output Parser
    - Datatime Output Parser [Now not available with Langchain v1]
    - Structured Output Parser (Pydanitc or Json)


### `Pydantinc` Output Parser

In [1]:
from dotenv import load_dotenv

load_dotenv('./../.env')

# langfuse or opik

True

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate,
                                        PromptTemplate
                                        )

base_url = "http://localhost:11434"
model = 'qwen3'

llm = ChatOllama(base_url=base_url, model=model)

In [3]:
from typing import  Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


In [4]:
class Joke(BaseModel):
    """Joke to tell user"""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(description="The rating of the joke is from 1 to 10", default=None)

In [5]:
parser = PydanticOutputParser(pydantic_object=Joke)

In [6]:
instruction = parser.get_format_instructions()

In [7]:
print(instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Joke to tell user", "properties": {"setup": {"description": "The setup of the joke", "title": "Setup", "type": "string"}, "punchline": {"description": "The punchline of the joke", "title": "Punchline", "type": "string"}, "rating": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "The rating of the joke is from 1 to 10", "title": "Rating"}}, "required": ["setup", "punchline"]}
```


In [8]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a joke. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = prompt | llm

In [9]:
output = chain.invoke({'query': 'Tell me a joke about the cat'})

In [10]:
print(output.content)

{
  "setup": "Why did the cat bring a ladder to the party?",
  "punchline": "Because it heard the food was on the top shelf!",
  "rating": 7
}


In [11]:
chain = prompt | llm | parser
output = chain.invoke({'query': 'Tell me a joke about the dogs'})
print(output)

setup='Why did the dog bring a ladder to the park?' punchline="To reach the top of the hill—it's the only way to get to the top of the world!" rating=8


### Parsing with `.with_structured_output()` method
- This method takes a schema as input which specifies the names, types, and descriptions of the desired output attributes.
-  The schema can be specified as a TypedDict class, JSON Schema or a Pydantic class.


In [12]:
output = llm.invoke('Tell me a joke about the cat')
print(output.content)

Why did the cat join a band?  
Because it wanted to be in the *meow-sic*! 🎵🐾  

*(Bonus: The band’s name was *Whisker & The Purr-fect Pitch*.)*


In [13]:
structured_llm = llm.with_structured_output(Joke)

In [14]:
output = structured_llm.invoke('Tell me a joke about the cat')
print(output)

setup='Why did the cat break the laser pointer?' punchline='It wanted to play with the real thing!' rating=None


### `JSON` Output Parser

- Output parsers accept a string or BaseMessage as input and can return an arbitrary type.



In [15]:
from langchain_core.output_parsers import JsonOutputParser

In [16]:
parser = JsonOutputParser(pydantic_object=Joke)
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [19]:
prompt = PromptTemplate(
    template='''
    Answer the user query with a joke. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = prompt | llm
output = chain.invoke({'query': 'Tell me a joke about the cat'})
print(output.content)

{"setup": "Why did the cat bring a ladder to the keyboard?", "punchline": "Because it wanted to reach the 'Enter' key!", "rating": 8}


In [18]:
chain = prompt | llm | parser
output = chain.invoke({'query': 'Tell me a joke about the cat'})
print(output)

{'setup': 'Why did the cat bring a ladder to the party?', 'punchline': 'It heard the drinks were on the house... and the snacks were on the floor!', 'rating': 8}


### CSV Output Parser

- This output parser can be used when you want to return a list of comma-separated items.



In [20]:
# value1, values2, values3, so on

from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

print(parser.get_format_instructions())

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [21]:
format_instruction = parser.get_format_instructions()

prompt = PromptTemplate(
    template='''
    Answer the user query with a list of values. Here is your formatting instruction.
    {format_instruction}

    Query: {query}
    Answer:''',
    input_variables=['query'],
    partial_variables={'format_instruction': format_instruction}
)   

In [22]:
chain = prompt | llm | parser

output = chain.invoke({'query': 'generate my website seo keywords. I have content about the NLP and LLM.'})
print(output)

['Natural Language Processing', 'Large Language Model', 'AI technology', 'Machine learning', 'Deep learning', 'Neural networks', 'Transformer models', 'BERT model', 'Text analysis', 'Chatbots', 'Sentiment analysis', 'Language models', 'Text generation', 'Summarization', 'Speech recognition', 'Translation', 'Data preprocessing', 'NLP applications', 'AI research', 'Technology trends', 'NLP techniques', 'LLM training', 'Language understanding', 'Semantic analysis', 'NLP tools', 'LLM frameworks', 'AI advancements', 'Text classification', 'Named entity recognition', 'Conversational AI', 'NLP pipelines', 'Language generation', 'AI ethics', 'NLP trends', 'LLM deployment', 'AI innovation', 'Text mining', 'NLP optimization', 'Language models', 'AI integration', 'NLP research', 'LLM capabilities', 'AI solutions', 'Text processing', 'NLP frameworks', 'Language understanding', 'AI development', 'NLP tools', 'LLM applications', 'AI trends', 'Text analysis', 'NLP techniques', 'LLM training', 'AI res